# Content refresh prioritisation — reproducible capstone

Prajwal Ravi Devadiga · FlyRank ML Internship

Run all cells to regenerate the metrics, report and published paper. Only aggregate outputs are saved.

## Abstract

Compare a transparent baseline with three classifiers for ranking pages by observed search-impression decline. Results are computed below, never typed into the narrative. This is a current-state proxy evaluation.

## Research question

Can search and content signals improve precision among the top 50 pages selected for human review? The capacity of 50 is an illustrative review budget, not a measured business outcome.

In [1]:
import os, sys
while not os.path.isdir("data/raw") and os.getcwd() != "/":
    os.chdir("..")

import pandas as pd, numpy as np
sys.path.insert(0, os.path.join(os.getcwd(), "scripts"))
from ml_utils import MODEL_NUMERIC_FEATURES, MODEL_CATEGORICAL_FEATURES, percentile_rank

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
fv = df[(df["impressions_90d"] > 0) & (df["content_age_days"] >= 90)].drop_duplicates("content_id").copy()

for col in ["impressions_90d", "clicks_90d", "sessions_90d", "ai_sessions_90d"]:
    fv[f"log_{col}"] = np.log1p(fv[col].fillna(0))
fv["is_declining_label"] = fv["trend_direction"].str.lower().eq("down").astype(int)
for col in MODEL_NUMERIC_FEATURES:
    fv[col] = pd.to_numeric(fv[col], errors="coerce").replace([np.inf, -np.inf], np.nan).fillna(0)
for col in MODEL_CATEGORICAL_FEATURES:
    fv[col] = fv[col].fillna("unknown").astype(str)

X = pd.get_dummies(fv[MODEL_NUMERIC_FEATURES + MODEL_CATEGORICAL_FEATURES], columns=MODEL_CATEGORICAL_FEATURES)
y = fv["is_declining_label"]

n_clients = fv["client_id"].nunique()
base_rate = y.mean()
median_pages = fv.groupby("client_id").size().median()
print(f"Dataset: {len(fv)} pages | {n_clients} clients | declining base rate: {base_rate:.1%}")
print(f"Editor capacity context: median client has {int(median_pages)} pages → ~{median_pages/50:.1f} weeks to review at 50/week")


Dataset: 30000 pages | 32 clients | declining base rate: 54.2%
Editor capacity context: median client has 567 pages → ~11.3 weeks to review at 50/week


## Data contract

Exclude direct target ingredients and IDs from model inputs. The 90-day features still overlap the target window; this is not a future-outcome experiment.

In [2]:
# Data contract verification
n_raw = len(df)
n_filt = len(fv)
n_dup = n_raw - len(df.drop_duplicates("content_id"))
n_imp_pos = (fv["impressions_90d"] > 0).sum()
n_age_ok = (fv["content_age_days"] >= 90).sum()

forbidden = ["trend_direction", "trend_pct", "impressions_last_30d", "clicks_last_30d",
             "sessions_last_30d", "impressions_prev_30d", "clicks_prev_30d",
             "sessions_prev_30d", "content_id", "client_id"]
model_cols = MODEL_NUMERIC_FEATURES + MODEL_CATEGORICAL_FEATURES
leaked = [c for c in forbidden if c in model_cols]

client_sizes = fv.groupby("client_id").size()

print("Data contract verification:")
print(f"  Total rows after filtering:       {n_filt}")
print(f"  Duplicate content_ids removed:        {n_dup}")
print(f"  Pages with impressions_90d > 0:   {n_imp_pos} ({n_imp_pos/n_filt:.1%})")
print(f"  Pages with content_age_days >= 90: {n_age_ok} ({n_age_ok/n_filt:.1%})")
print(f"\nExcluded leakage columns in model features: {len(leaked)}  ✓")
print(f"Label base rate: {y.mean():.1%}  ({int(y.sum())} declining of {len(y)})")
print(f"\nClient size distribution:")
print(f"  min: {client_sizes.min():>4} pages | median: {int(client_sizes.median()):>4} | max: {client_sizes.max()}")
print(f"  {fv['client_id'].nunique()} distinct pseudonymised clients")
assert not set(MODEL_NUMERIC_FEATURES + MODEL_CATEGORICAL_FEATURES) & {"trend_direction", "trend_pct", "content_id", "client_id", "impressions_last_30d", "impressions_prev_30d", "clicks_last_30d", "clicks_prev_30d", "sessions_last_30d", "sessions_prev_30d"}
assert fv["content_id"].is_unique


Data contract verification:
  Total rows after filtering:       30000
  Duplicate content_ids removed:        0
  Pages with impressions_90d > 0:   30000 (100.0%)
  Pages with content_age_days >= 90: 30000 (100.0%)

Excluded leakage columns in model features: 0  ✓
Label base rate: 54.2%  (16262 declining of 30000)

Client size distribution:
  min:    3 pages | median:  567 | max: 7008
  32 distinct pseudonymised clients


## Client-grouped comparison

Hold out six clients using seed 42. Category discovery uses the full unlabelled snapshot. These are exploratory development results, not a sealed test.

In [3]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return float(np.asarray(labels)[order[:min(k, len(labels))]].mean())

# Client-grouped split, seed=42
rng = np.random.default_rng(42)
unique_clients = fv["client_id"].drop_duplicates().to_numpy()
shuffled = rng.permutation(unique_clients)
n_holdout = max(1, int(round(len(shuffled) * 0.2)))
holdout_clients = set(shuffled[:n_holdout])
is_test = fv["client_id"].isin(holdout_clients).to_numpy()

X_train, X_test = X[~is_test], X[is_test]
y_train, y_test = y[~is_test], y[is_test]

rf = RandomForestClassifier(class_weight="balanced_subsample", max_depth=10,
                             min_samples_leaf=25, n_estimators=200, n_jobs=-1, random_state=42)
rf.fit(X_train, y_train)

lr = Pipeline([("scaler", StandardScaler()),
               ("model", LogisticRegression(class_weight="balanced", max_iter=1000, random_state=42))])
lr.fit(X_train, y_train)

dt = DecisionTreeClassifier(class_weight="balanced", max_depth=5, min_samples_leaf=50, random_state=42)
dt.fit(X_train, y_train)

print(f"Model training complete.")
print(f"  Train rows: {len(X_train)} ({len(unique_clients)-n_holdout} clients)")
print(f"  Test rows:  {len(X_test)}  ({n_holdout} holdout clients)")
print(f"  Test base rate: {y_test.mean():.3f}")
print(f"  Feature columns: {X.shape[1]}")
assert set(fv.loc[is_test, "client_id"]).isdisjoint(set(fv.loc[~is_test, "client_id"]))
assert y_test.nunique() == 2 and y_train.nunique() == 2


Model training complete.
  Train rows: 27675 (26 clients)
  Test rows:  2325  (6 holdout clients)
  Test base rate: 0.391
  Feature columns: 52


## Transparent baseline

Use 45% visibility percentile, 35% staleness percentile and 20% positive CTR-gap percentile. Ranks and position-tier CTR medians use the full snapshot before scoring the same holdout rows as the models.

In [4]:
# Full-snapshot baseline, evaluated on held-out clients
fv["visibility_score"] = fv["impressions_90d"].rank(pct=True)
fv["freshness_risk_score"] = fv["days_since_last_update"].rank(pct=True)
positioned = fv[(fv["avg_position"] > 0) & (fv["avg_position"] <= 50)]
tier_med = positioned.groupby("position_tier")["ctr"].median()
fv["exp_ctr"] = fv["position_tier"].map(tier_med).fillna(fv["ctr"])
fv["ctr_gap"] = (fv["exp_ctr"] - fv["ctr"]).clip(lower=0)
fv["ctr_gap_score"] = fv["ctr_gap"].rank(pct=True) * (fv["avg_position"] > 0).astype(int)
fv["baseline_score"] = (0.45 * fv["visibility_score"] + 0.35 * fv["freshness_risk_score"]
                        + 0.20 * fv["ctr_gap_score"]).clip(0, 1)
p_base = fv.loc[is_test, "baseline_score"].to_numpy()

p_rf  = rf.predict_proba(X_test)[:, 1]
p_lr  = lr.predict_proba(X_test)[:, 1]
p_dt  = dt.predict_proba(X_test)[:, 1]

base_p50 = precision_at_k(p_base, y_test.values, 50)

print(f"Results on client-grouped holdout (seed=42, {len(y_test)} test rows, base rate={y_test.mean():.3f})\n")
print(f"{'method':<29} {'ROC-AUC':>8} {'P@20':>7} {'P@50':>7}  {'lift_vs_baseline'}")
print("-" * 71)

for name, p, show_lift in [
    ("baseline_action_score",  p_base, False),
    ("logistic_regression",    p_lr,   True),
    ("decision_tree (d=5)",    p_dt,   True),
    ("random_forest",          p_rf,   True),
]:
    auc = roc_auc_score(y_test, p)
    p20 = precision_at_k(p, y_test.values, 20)
    p50 = precision_at_k(p, y_test.values, 50)
    lift = f"+{p50-base_p50:.3f}" if (show_lift and p50 > base_p50) else (f"{p50-base_p50:.3f}" if show_lift else "--")
    print(f"{name:<29} {auc:>8.3f} {p20:>7.3f} {p50:>7.3f}  {lift:>16}")

print(f"\nTest base rate (floor):        {y_test.mean():.3f}")
print(f"Seed-42 is the canonical reported number for this repo.")


Results on client-grouped holdout (seed=42, 2325 test rows, base rate=0.391)

method                         ROC-AUC    P@20    P@50  lift_vs_baseline
-----------------------------------------------------------------------
baseline_action_score            0.646   0.400   0.420                --
logistic_regression              0.700   0.350   0.400            -0.020
decision_tree (d=5)              0.742   0.650   0.660            +0.240
random_forest                    0.747   0.700   0.680            +0.260

Test base rate (floor):        0.391
Seed-42 is the canonical reported number for this repo.


## Measured split sensitivity

Refit the Random Forest for each client-split seed. Keep forest random_state=42 fixed to isolate split sensitivity. Compare each holdout against its own baseline and base rate.

In [5]:
from sklearn.base import clone
seed_results = []
for split_seed in [42, 7, 123]:
    clients = np.random.default_rng(split_seed).permutation(unique_clients)[:n_holdout]
    mask = fv["client_id"].isin(clients).to_numpy()
    assert set(fv.loc[mask, "client_id"]).isdisjoint(set(fv.loc[~mask, "client_id"]))
    model = rf if split_seed == 42 else clone(rf).fit(X[~mask], y[~mask])
    scores = model.predict_proba(X[mask])[:, 1]
    seed_results.append({"split_seed": split_seed, "test_rows": int(mask.sum()),
        "test_base_rate": round(float(y[mask].mean()), 3),
        "rf_precision_at_50": round(precision_at_k(scores, y[mask], 50), 3),
        "baseline_precision_at_50": round(precision_at_k(fv.loc[mask, "baseline_score"], y[mask], 50), 3)})
print(pd.DataFrame(seed_results).to_string(index=False))


 split_seed  test_rows  test_base_rate  rf_precision_at_50  baseline_precision_at_50
         42       2325           0.391                0.68                      0.42
          7       4280           0.611                0.84                      0.76
        123       3090           0.549                0.98                      0.98


## Demonstration action queue

Scores below include training rows. Action rules also use the observed trend label. This queue demonstrates a review workflow; it is not independent evaluation evidence and row-level data is not exported.

In [6]:
# Generate the full queue
def assign_action(row):
    wc = row.get("word_count", 0) or 0
    imp = row.get("impressions_90d", 0) or 0
    pos = row.get("avg_position", 0) or 0
    ctr = row.get("ctr", 0) or 0
    stale = row.get("days_since_last_update", 0) or 0
    trend = str(row.get("trend_direction", "")).lower()
    if wc > 0 and wc < 1000 and imp >= 500:
        return "expand_and_refresh"
    if pos > 0 and pos <= 20 and ctr < 0.3 and imp >= 300:
        return "refresh_and_review_ctr"
    if trend == "down" and imp >= 100:
        return "refresh"
    if stale >= 180 and imp >= 100:
        return "refresh"
    return "monitor"

fv["rf_score"] = rf.predict_proba(X)[:, 1]
fv["action"] = fv.apply(assign_action, axis=1)
fv["queue_rank"] = fv["rf_score"].rank(method="first", ascending=False).astype(int)
queue = fv.sort_values("queue_rank")

print("Demonstration queue summary (includes training rows):")
print(f"  {'action':<24} {'count':>6} {'pct':>7}  {'declining_rate_in_action'}")
for action, grp in queue.groupby("action"):
    cnt = len(grp)
    dec_rate = grp["is_declining_label"].mean()
    print(f"  {action:<24} {cnt:>6} {cnt/len(queue):>6.1%}  {dec_rate:>10.3f}")



Demonstration queue summary (includes training rows):
  action                    count     pct  declining_rate_in_action
  expand_and_refresh            9   0.0%       0.222
  monitor                   13934  46.4%       0.223
  refresh                    7655  25.5%       0.999
  refresh_and_review_ctr     8402  28.0%       0.655


## Canonical aggregate outputs

Generate the report, public paper and README status from the metrics computed in this run. Record environment and input provenance.

In [7]:
import pathlib, json

out_dir = pathlib.Path("work/outputs")
out_dir.mkdir(parents=True, exist_ok=True)

# 1. Exploratory holdout metrics
metrics = {
    "split": "client_grouped_seed42",
    "n_clients_total": int(fv["client_id"].nunique()),
    "n_holdout_clients": int(n_holdout),
    "train_rows": int(len(X_train)),
    "test_rows": int(len(X_test)),
    "test_base_rate": round(float(y_test.mean()), 3),
    "label_base_rate_full": round(float(y.mean()), 3),
    "random_forest": {
        "roc_auc": round(roc_auc_score(y_test, p_rf), 3),
        "precision_at_20": round(precision_at_k(p_rf, y_test.values, 20), 3),
        "precision_at_50": round(precision_at_k(p_rf, y_test.values, 50), 3),
    },
    "decision_tree": {
        "roc_auc": round(roc_auc_score(y_test, p_dt), 3),
        "precision_at_20": round(precision_at_k(p_dt, y_test.values, 20), 3),
        "precision_at_50": round(precision_at_k(p_dt, y_test.values, 50), 3),
    },
    "logistic_regression": {
        "roc_auc": round(roc_auc_score(y_test, p_lr), 3),
        "precision_at_20": round(precision_at_k(p_lr, y_test.values, 20), 3),
        "precision_at_50": round(precision_at_k(p_lr, y_test.values, 50), 3),
    },
    "baseline": {
        "roc_auc": round(roc_auc_score(y_test, p_base), 3),
        "precision_at_20": round(precision_at_k(p_base, y_test.values, 20), 3),
        "precision_at_50": round(precision_at_k(p_base, y_test.values, 50), 3),
    },
}
import hashlib, platform, importlib.metadata
metrics["seed_sensitivity"] = seed_results
metrics["provenance"] = {
    "dataset_sha256": hashlib.sha256(pathlib.Path("data/raw/content_refresh_anonymized.csv").read_bytes()).hexdigest(),
    "feature_code_sha256": hashlib.sha256(pathlib.Path("scripts/ml_utils.py").read_bytes()).hexdigest(),
    "notebook_code_sha256": hashlib.sha256("\n".join("".join(c["source"]) for c in json.loads(pathlib.Path("work/notebooks/capstone.ipynb").read_text())["cells"] if c["cell_type"] == "code").encode()).hexdigest(),
    "python": platform.python_version(),
    "packages": {name: importlib.metadata.version(name) for name in ["numpy", "pandas", "scipy", "scikit-learn"]},
    "rf_parameters": rf.get_params(),
    "preprocessing": "Full-snapshot category discovery, baseline percentile ranks and CTR medians; numeric missing values filled with zero.",
    "ranking": "Descending numpy.argsort, default tie ordering; pinned NumPy version.",
    "scope": "Exploratory current-state proxy; overlapping windows; not a sealed or prospective evaluation."
}
metrics["feature_importance"] = [{"feature": str(k), "importance": round(float(v), 4)} for k,v in pd.Series(rf.feature_importances_, index=X.columns).nlargest(10).items()]
metrics["label_distribution"] = {str(k): int(v) for k,v in fv["trend_direction"].value_counts().items()}
assert metrics["seed_sensitivity"][0]["rf_precision_at_50"] == metrics["random_forest"]["precision_at_50"]
(out_dir / "capstone_model_metrics.json").write_text(json.dumps(metrics, indent=2) + "\n")
from render_capstone_report import render
render(pathlib.Path.cwd())
print("Generated aggregate metrics, Markdown report, public paper and README status.")


Generated aggregate metrics, Markdown report, public paper and README status.


## Interpretation and next experiment

Use the generated report for current numbers. Feature importance is descriptive, not causal. Before deployment, fit all preprocessing on training clients, collect non-overlapping future outcomes and use a new untouched evaluation set.

## Acknowledgments

Dataset and starter foundation courtesy of [FlyRank](https://flyrank.ai). Earlier weekly notebooks are historical experiments and may use different baselines or environments; this capstone is the canonical report.